# Week 2 — Leakage-safe Titanic baseline

The split occurs before fitting imputers or encoders. The preprocessing and classifier are kept in one scikit-learn pipeline so test-set statistics are not used during training.

In [ ]:
from pathlib import Path
import os

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier


In [ ]:
RANDOM_STATE = 42
DATA_PATH = Path(
    os.environ.get("TITANIC_TRAIN_CSV", "data/titanic/train.csv")
).expanduser()
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Titanic training data was not found at {DATA_PATH}. "
        "Set TITANIC_TRAIN_CSV to the absolute path of train.csv."
    )

df = pd.read_csv(DATA_PATH)
required_columns = {"Survived", "Age", "Embarked", "Sex"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(
        f"Dataset is missing required columns: {sorted(missing_columns)}. "
        f"Found: {sorted(df.columns.tolist())}"
    )
if df.empty:
    raise ValueError(f"Dataset at {DATA_PATH} is empty.")


In [ ]:
# Remove exact duplicate records before splitting so the same record cannot
# appear in both train and test partitions.
df = df.drop_duplicates().reset_index(drop=True)
y = df["Survived"]
if y.isna().any():
    raise ValueError("Survived contains missing labels; clean labels before training.")
if y.nunique() != 2:
    raise ValueError(f"Expected a binary Survived target, found {y.nunique()} classes.")
if y.value_counts().min() < 2:
    raise ValueError("Each target class needs at least two rows for a stratified split.")

# Identifiers and free text are intentionally excluded from this baseline.
X = df.drop(columns=["Survived", "Name", "Ticket", "Cabin", "PassengerId"], errors="ignore")
if X.shape[1] == 0:
    raise ValueError("No usable feature columns remain after exclusions.")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

categorical_features = [
    column for column in ("Sex", "Embarked") if column in X_train.columns
]
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
transformers = []
if numeric_features:
    transformers.append(
        ("numeric", SimpleImputer(strategy="median"), numeric_features)
    )
if categorical_features:
    transformers.append(
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_features,
        )
    )
if not transformers:
    raise ValueError("No numeric or categorical features are available for preprocessing.")


In [ ]:
model = Pipeline(
    steps=[
        ("preprocessor", ColumnTransformer(transformers=transformers)),
        ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)


In [ ]:
print(f"Rows: train={len(X_train):,}, test={len(X_test):,}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-score: {f1_score(y_test, y_pred, zero_division=0):.4f}")
